<a href="https://colab.research.google.com/github/kojeda603/analisis_exploratorio_de_datos/blob/main/14_PARCIAL3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3er Examen Parcial — Limpieza y transformación de datos con pandas
**Estudiante:** Ojeda Pardo  
**Asignatura:** Análisis Exploratorio de Datos  
**Dataset:** prescripciones_raw_ojeda_pardo.csv  


In [198]:
# Importación de librerías necesarias para la limpieza
import pandas as pd
import numpy as np
import re
from google.colab import drive
from google.colab import files

# Cargar el dataset en un DataFrame
drive.mount("/content/drive")
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/ciencia_de_datos/prescripciones_raw_ojeda_pardo.csv")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Nivel 1 — Diagnóstico

### 1. Inspección inicial: forma, tipos, columnas y primeras filas

In [199]:
# Forma del dataset (filas, columnas)
print("Forma del dataset:", df.shape)
print("\nNombres de columnas:")
print(df.columns.tolist())


# Tipos de dato por columna
print("\nTipos de dato:")
print(df.dtypes)

Forma del dataset: (520, 10)

Nombres de columnas:
['id_paciente', 'fecha_prescripcion', 'medicamento', 'dosis_mg', 'via_administracion', 'diagnostico_cie10', 'peso_paciente_kg', 'edad', 'eps', 'prescriptor_id']

Tipos de dato:
id_paciente             int64
fecha_prescripcion     object
medicamento            object
dosis_mg               object
via_administracion     object
diagnostico_cie10      object
peso_paciente_kg      float64
edad                  float64
eps                    object
prescriptor_id         object
dtype: object


In [200]:
# Información compacta del DataFrame
print("\n--- df.info() ---")
df.info()

# Estadística descriptiva de variables numéricas
print("\n--- df.describe() ---")
print(df.describe(include='all'))



--- df.info() ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 520 entries, 0 to 519
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_paciente         520 non-null    int64  
 1   fecha_prescripcion  520 non-null    object 
 2   medicamento         520 non-null    object 
 3   dosis_mg            489 non-null    object 
 4   via_administracion  520 non-null    object 
 5   diagnostico_cie10   403 non-null    object 
 6   peso_paciente_kg    500 non-null    float64
 7   edad                505 non-null    float64
 8   eps                 520 non-null    object 
 9   prescriptor_id      464 non-null    object 
dtypes: float64(2), int64(1), object(7)
memory usage: 40.8+ KB

--- df.describe() ---
        id_paciente fecha_prescripcion  medicamento dosis_mg  \
count    520.000000                520          520      489   
unique          NaN                458           11       22   
top       

In [201]:
# Primeras 15 filas
df.head(15)

,id_paciente,fecha_prescripcion,medicamento,dosis_mg,via_administracion,diagnostico_cie10,peso_paciente_kg,edad,eps,prescriptor_id
0,1382,25/12/2023,Amoxicilina,1000.0,Oral,NaN,76.160869,50.0,SURA,MED-4787
1,8182,2023-05-26,METFORMINA,200.0,VO,diabetes tipo 2,68.535582,76.0,Sanitas,MED-8065
2,5669,18/02/2025,ASA,500.0,intravenosa,diabetes tipo 2,81.009696,NaN,EPS Sanitas,MED-3701
3,2303,2024-11-19,Aspirina,400.0,VO,hipertension,57.342877,78.0,EPS Sanitas,MED-6604
4,2245,30/09/2024,Aspirina,400.0,IV,J18.9,58.150958,31.0,Sura,MED-9233
5,3113,2026-03-26,METFORMINA,250.0,via oral,NaN,80.086736,82.0,Compensar,MED-1887
6,3492,2025-05-27,Aspirina,850.0,Subcutanea,I10,64.456528,52.0,COMPENSAR,MED-0487
7,6358,2023-05-29,METFORMINA,NaN,via oral,diabetes tipo 2,60.109236,80.0,Sura,MED-1273
8,6825,diciembre 4 2024,Amoxicilina,500.0,Oral,hipertension,3.000000,31.0,SURA,MED-0488
9,5637,2023-08-26,METFORMINA,200.0,VO,I10,64.630214,45.0,SANITAS,MED-1056


### 2. Inventario de valores faltantes
Se calcula el número y porcentaje de valores nulos por columna, ordenado de mayor a menor cantidad de faltantes (es decir, de menor a mayor completitud).

In [202]:
# Conteo y porcentaje de NaN por columna
faltantes = df.isnull().sum()
porcentaje = (df.isnull().sum() / len(df) * 100).round(2)

# Tabla resumen de completitud
inventario_na = pd.DataFrame({
    'n_faltantes': faltantes,
    'porcentaje_%': porcentaje,
    'completitud_%': (100 - porcentaje).round(2)
}).sort_values('n_faltantes', ascending=False)

inventario_na

,n_faltantes,porcentaje_%,completitud_%
diagnostico_cie10,117,22.50,77.50
prescriptor_id,56,10.77,89.23
dosis_mg,31,5.96,94.04
peso_paciente_kg,20,3.85,96.15
edad,15,2.88,97.12
id_paciente,0,0.00,100.00
fecha_prescripcion,0,0.00,100.00
medicamento,0,0.00,100.00
via_administracion,0,0.00,100.00
eps,0,0.00,100.00


### 3. Documentación de problemas detectados por columna

| Columna | Tipo de problema |
|---|---|
| **id_paciente** | Duplicados parciales (mismo paciente con varias prescripciones distintas y duplicados exactos) |
| **fecha_prescripcion** | Tipo incorrecto (object) y formatos diferentes como: `YYYY-MM-DD`, `DD/MM/YYYY` y texto en español como `25/12/2023` |
| **medicamento** | Inconsistencia categórica: variaciones en mayúsculas, espacios sobrantes y sinónimos (`ASA` ≡ `Aspirina`), variantes con dosis incrustada (`Metformina 500`) |
| **dosis_mg** | Tipo mixto (números, strings tipo `100mg`, valores negativos, valores extremos como `50000`) y NaN |
| **via_administracion** | Inconsistencia categórica: múltiples formas para la misma vía (`VO`, `oral`, `Oral`, `via oral`) |
| **diagnostico_cie10** | Mezcla de códigos CIE-10  y NaN |
| **peso_paciente_kg** | Outliers fisiológicamente imposibles y NaN |
| **edad** | Valores inválidos (negativos, 0, 200) y NaN |
| **eps** | Inconsistencia categórica: variantes para una misma EPS (`SURA`, `Sura`, `EPS Sura`) |
| **prescriptor_id** | NaN y formato variable |

**Justificación general:** las inconsistencias provienen de captura manual y exportaciones de varias fuentes; cada problema requiere una estrategia distinta (normalización, imputación o eliminación) tomando en cuenta el contexto clínico.

## Nivel 2 — Limpieza

### 4. Estandarización de `fecha_prescripcion`
El campo trae tres formatos distintos. La estrategia es:
Reemplazar nombres de meses en español por su número equivalente y reorganizar los formatos.

In [203]:
# Diccionario de meses en español -> número
meses_es = {
    'enero': '01', 'febrero': '02', 'marzo': '03', 'abril': '04',
    'mayo': '05', 'junio': '06', 'julio': '07', 'agosto': '08',
    'septiembre': '09', 'octubre': '10', 'noviembre': '11', 'diciembre': '12'
}

def convertir_fecha_es(valor):
    """Convierte texto tipo 'diciembre 4 2024' a '2024-12-04'."""
    if not isinstance(valor, str):
        return valor
    texto = valor.strip().lower()
    # Detecta patrón: mes dia año
    for mes_nombre, mes_numero in meses_es.items():
        if mes_nombre in texto:
            partes = texto.replace(mes_nombre, '').strip().split()
            if len(partes) != 2:
                return valor

            # Identificar día y año por longitud del string
            if len(partes[0]) == 4 and partes[0].isdigit():
                año, dia = partes[0], partes[1]
            elif len(partes[1]) == 4 and partes[1].isdigit():
                dia, año = partes[0], partes[1]
            else:
                return valor  # formato no reconocible

            # Validar que día sea numérico y esté en rango
            if not dia.isdigit() or not (1 <= int(dia) <= 31):
                return valor

            return f"{año}-{mes_numero}-{int(dia):02d}"

    return valor

# Guardar copia del original (para auditar después)
df['fecha_original'] = df['fecha_prescripcion'].copy()

# Pasada 0: normalizar los textos en español a formato ISO
texto_normalizado = df['fecha_original'].apply(convertir_fecha_es)

# Pasada 1: parsear formato ISO (YYYY-MM-DD)
fecha_iso = pd.to_datetime(texto_normalizado, format='%Y-%m-%d', errors='coerce')

# Pasada 2: parsear formato DMA (DD/MM/YYYY) para los que fallaron en ISO
fecha_dma = pd.to_datetime(texto_normalizado, format='%d/%m/%Y', errors='coerce')

# Combinar: usa ISO donde funcionó, y DMA donde ISO falló
df['fecha_prescripcion'] = fecha_iso.fillna(fecha_dma)

# ---------- Auditoría de calidad ----------

total = len(df)
ok = df['fecha_prescripcion'].notna().sum()
nat = df['fecha_prescripcion'].isna().sum()

print(f"=== Resultado de conversión ===")
print(f"Total filas:      {total}")
print(f"Convertidas OK:   {ok} ({ok/total*100:.1f}%)")
print(f"NaT restantes:    {nat} ({nat/total*100:.1f}%)")

if nat > 0:
    print(f"\n--- Top 10 patrones que aún fallan ---")
    fallidas = df.loc[df['fecha_prescripcion'].isna(), 'fecha_original']
    print(fallidas.value_counts().head(10))

print(f"\n--- Muestra de conversiones exitosas ---")
muestra = df.loc[df['fecha_prescripcion'].notna(), ['fecha_original', 'fecha_prescripcion']].head(10)
print(muestra)

=== Resultado de conversión ===
Total filas:      520
Convertidas OK:   520 (100.0%)
NaT restantes:    0 (0.0%)

--- Muestra de conversiones exitosas ---
     fecha_original fecha_prescripcion
0        25/12/2023         2023-12-25
1        2023-05-26         2023-05-26
2        18/02/2025         2025-02-18
3        2024-11-19         2024-11-19
4        30/09/2024         2024-09-30
5        2026-03-26         2026-03-26
6        2025-05-27         2025-05-27
7        2023-05-29         2023-05-29
8  diciembre 4 2024         2024-12-04
9        2023-08-26         2023-08-26


### 5. Normalización de `medicamento` y `via_administracion`
Se aplica `strip` y `lower`, y se mapean equivalentes a una forma canónica.  
- En **medicamento**, agrupo `ASA` y `Aspirina` (son sinónimos del mismo principio activo: ácido acetilsalicílico). `Metformina 500` se unifica con `metformina`.  
- En **via_administracion**, `VO`, `oral` y `via oral` → `oral`; `IV` e `intravenosa` → `intravenosa`; `SC` y `subcutanea` → `subcutanea`.

In [204]:
print("=== DIAGNÓSTICO INICIAL ===")
print(f"\nMedicamento — {df['medicamento'].nunique()} valores únicos:")
print(df['medicamento'].value_counts(dropna=False))

print(f"\nVía de administración — {df['via_administracion'].nunique()} valores únicos:")
print(df['via_administracion'].value_counts(dropna=False))

=== DIAGNÓSTICO INICIAL ===

Medicamento — 11 valores únicos:
medicamento
amoxicilina       62
Enalapril         57
METFORMINA        55
metformina        49
Ibuprofeno        48
ibuprofeno        47
ASA               45
Amoxicilina       44
Aspirina          44
ENALAPRIL         44
Metformina 500    25
Name: count, dtype: int64

Vía de administración — 10 valores únicos:
via_administracion
IV             71
intravenosa    63
Subcutanea     57
Intravenosa    54
subcutanea     52
Oral           50
via oral       50
oral           46
VO             39
SC             38
Name: count, dtype: int64


In [205]:
# Guardar copias originales para auditoría
df['medicamento_original'] = df['medicamento'].copy()
df['via_original'] = df['via_administracion'].copy()

# Normalización básica: a string, sin espacios extra, en minúsculas
df['medicamento'] = df['medicamento'].astype(str).str.strip().str.lower()
df['via_administracion'] = df['via_administracion'].astype(str).str.strip().str.lower()

# Tratar nulos disfrazados como NaN reales
nulos_disfrazados = ['nan', '', 'n/a', 'na', '-', 'sin dato', 'null', 'none']
df['medicamento'] = df['medicamento'].replace(nulos_disfrazados, np.nan)
df['via_administracion'] = df['via_administracion'].replace(nulos_disfrazados, np.nan)

In [206]:
print("=== DESPUÉS DE NORMALIZACIÓN BÁSICA ===")
print(f"\nMedicamento — {df['medicamento'].nunique()} valores únicos:")
print(df['medicamento'].value_counts(dropna=False))

print(f"\nVía de administración — {df['via_administracion'].nunique()} valores únicos:")
print(df['via_administracion'].value_counts(dropna=False))

=== DESPUÉS DE NORMALIZACIÓN BÁSICA ===

Medicamento — 7 valores únicos:
medicamento
amoxicilina       106
metformina        104
enalapril         101
ibuprofeno         95
asa                45
aspirina           44
metformina 500     25
Name: count, dtype: int64

Vía de administración — 7 valores únicos:
via_administracion
intravenosa    117
subcutanea     109
oral            96
iv              71
via oral        50
vo              39
sc              38
Name: count, dtype: int64


In [207]:
# Diccionario construido a partir de los valores REALES vistos en el diagnóstico
mapeo_medicamento = {
    'asa': 'aspirina',
    'metformina 500': 'metformina',
}

mapeo_via = {
    'vo': 'oral',
    'via oral': 'oral',
    'iv': 'intravenosa',
    'sc': 'subcutanea',
}

df['medicamento'] = df['medicamento'].replace(mapeo_medicamento)
df['via_administracion'] = df['via_administracion'].replace(mapeo_via)

In [208]:
print("=== RESULTADO FINAL ===")
print(f"\nMedicamento — {df['medicamento'].nunique()} valores únicos:")
print(df['medicamento'].value_counts(dropna=False))

print(f"\nVía de administración — {df['via_administracion'].nunique()} valores únicos:")
print(df['via_administracion'].value_counts(dropna=False))

# Comparativa: cuánto se redujo la cardinalidad
print(f"\n--- Reducción de cardinalidad ---")
print(f"Medicamento: {df['medicamento_original'].nunique()} → {df['medicamento'].nunique()}")
print(f"Vía: {df['via_original'].nunique()} → {df['via_administracion'].nunique()}")

=== RESULTADO FINAL ===

Medicamento — 5 valores únicos:
medicamento
metformina     129
amoxicilina    106
enalapril      101
ibuprofeno      95
aspirina        89
Name: count, dtype: int64

Vía de administración — 3 valores únicos:
via_administracion
intravenosa    188
oral           185
subcutanea     147
Name: count, dtype: int64

--- Reducción de cardinalidad ---
Medicamento: 11 → 5
Vía: 10 → 3


### 6. Limpieza de `dosis_mg`
- (a) Extracción del valor numérico de strings mixtos (`"100mg"` → `100`).
- (b) **Tratamiento de negativos:** una dosis negativa es físicamente imposible, indica error de digitación. Se reemplaza por `NaN` (no se conserva el valor absoluto porque el signo podría ocultar otro error).
- (c) **Imputación de NaN:** se usa la **mediana por medicamento** con `groupby` + `transform`. La mediana es robusta a outliers (hay valores como 50000 mg) y la imputación por grupo respeta la posología típica de cada fármaco.

In [209]:
# ============================================================
# Limpieza de dosis_mg: extracción + validación + imputación
# ============================================================

def extraer_dosis(valor):
    """Extrae el primer número de un string tipo '100mg' o '850 mg'."""
    if pd.isna(valor):
        return np.nan
    if isinstance(valor, (int, float)):
        return float(valor)
    coincidencia = re.search(r'-?\d+\.?\d*', str(valor))
    return float(coincidencia.group()) if coincidencia else np.nan

# Guardar copia del original (para auditar después)
df['dosis_original'] = df['dosis_mg'].copy()

# (a) Extraer numérico de strings mixtos
df['dosis_mg'] = df['dosis_mg'].apply(extraer_dosis)
nan_post_extraccion = df['dosis_mg'].isna().sum()

# (b) Reemplazar negativos por NaN (físicamente imposibles)
n_negativos = (df['dosis_mg'] < 0).sum()
df.loc[df['dosis_mg'] < 0, 'dosis_mg'] = np.nan

# (c) Reemplazar outliers con umbral clínico por medicamento
dosis_maxima = {
    'metformina': 2550,
    'amoxicilina': 3000,
    'aspirina': 1500,
    'losartan': 200,
    'omeprazol': 40
}
DOSIS_MAX_DEFAULT = 5000

clave_med = df['medicamento'].astype(str).str.strip().str.lower()
umbral = clave_med.map(dosis_maxima).fillna(DOSIS_MAX_DEFAULT)
mascara_extremos = df['dosis_mg'] > umbral
n_extremos = mascara_extremos.sum()
df.loc[mascara_extremos, 'dosis_mg'] = np.nan


# Métrica intermedia: NaN antes de imputar
nan_pre_imputacion = df['dosis_mg'].isna().sum()

# (d) Imputar NaN con mediana por medicamento
df['dosis_mg'] = df['dosis_mg'].fillna(
    df.groupby(clave_med)['dosis_mg'].transform('median')
)
nan_post_mediana_med = df['dosis_mg'].isna().sum()


# Si aún queda algún NaN (medicamento sin mediana válida), imputar con la mediana global
df['dosis_mg'] = df['dosis_mg'].fillna(df['dosis_mg'].median())

# ---------- Auditoría de calidad ----------

total = len(df)
ok = df['dosis_mg'].notna().sum()
nat_finales = df['dosis_mg'].isna().sum()

# Cuántos se imputaron en cada paso
imputados_por_medicamento = nan_pre_imputacion - nan_post_mediana_med
imputados_global = nan_post_mediana_med  # los que quedaron después de mediana por med

print(f"=== Resultado de limpieza de dosis_mg ===")
print(f"Total filas:                    {total}")
print(f"Procesadas OK:                  {ok} ({ok/total*100:.1f}%)")
print(f"NaN restantes:                  {nat_finales} ({nat_finales/total*100:.1f}%)")

print(f"\n--- Desglose del proceso ---")
print(f"NaN tras extraer número:        {nan_post_extraccion}")
print(f"Valores negativos -> NaN:       {n_negativos}")
print(f"NaN antes de imputar:           {nan_pre_imputacion}")
print(f"Imputados por mediana de med.:  {imputados_por_medicamento}")
print(f"Imputados por mediana global:   {imputados_global}")

print(f"\n--- Estadísticas finales ---")
print(df['dosis_mg'].describe())

print(f"\n--- Muestra de limpieza (con medicamento) ---")
muestra = df[['medicamento', 'dosis_original', 'dosis_mg']].head(10)
print(muestra)

=== Resultado de limpieza de dosis_mg ===
Total filas:                    520
Procesadas OK:                  520 (100.0%)
NaN restantes:                  0 (0.0%)

--- Desglose del proceso ---
NaN tras extraer número:        31
Valores negativos -> NaN:       15
NaN antes de imputar:           51
Imputados por mediana de med.:  51
Imputados por mediana global:   0

--- Estadísticas finales ---
count     520.000000
mean      462.788462
std       292.759302
min       100.000000
25%       200.000000
50%       400.000000
75%       850.000000
max      1000.000000
Name: dosis_mg, dtype: float64

--- Muestra de limpieza (con medicamento) ---
   medicamento dosis_original  dosis_mg
0  amoxicilina         1000.0    1000.0
1   metformina          200.0     200.0
2     aspirina          500.0     500.0
3     aspirina          400.0     400.0
4     aspirina          400.0     400.0
5   metformina          250.0     250.0
6     aspirina          850.0     850.0
7   metformina            NaN     40

In [210]:
print("\n--- Dosis por medicamento (después de imputar) ---")
for med in df['medicamento'].dropna().unique():
    dosis_med = df.loc[df['medicamento'] == med, 'dosis_mg']
    print(f"\n{med}: mediana={dosis_med.median()}, moda={dosis_med.mode().iloc[0]}")
    print(dosis_med.value_counts().sort_index().head(5))


--- Dosis por medicamento (después de imputar) ---

amoxicilina: mediana=400.0, moda=400.0
dosis_mg
100.0    11
200.0    15
250.0    10
400.0    27
500.0    16
Name: count, dtype: int64

metformina: mediana=400.0, moda=400.0
dosis_mg
100.0    17
200.0    19
250.0    10
400.0    31
500.0    15
Name: count, dtype: int64

aspirina: mediana=400.0, moda=400.0
dosis_mg
100.0    12
200.0     9
250.0    15
400.0    23
500.0     6
Name: count, dtype: int64

ibuprofeno: mediana=400.0, moda=400.0
dosis_mg
100.0    15
200.0    12
250.0     7
400.0    27
500.0    12
Name: count, dtype: int64

enalapril: mediana=400.0, moda=400.0
dosis_mg
100.0    11
200.0    15
250.0    10
400.0    32
500.0    11
Name: count, dtype: int64


### 7. Tratamiento de outliers en `peso_paciente_kg` (método IQR)
Calculo Q1, Q3 e IQR. Considero outlier todo valor fuera de `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`.  
**Decisión:** **imputar** (no eliminar) con la mediana. Eliminar registros descartaría información clínica útil (medicamento, dosis, edad) por un error puntual en una sola variable. La mediana es robusta a valores extremos.

In [211]:
# Distribución antes
print("Distribución ANTES del tratamiento de outliers:")
print(df['peso_paciente_kg'].describe())

# Cálculo de límites IQR
Q1 = df['peso_paciente_kg'].quantile(0.25)
Q3 = df['peso_paciente_kg'].quantile(0.75)
IQR = Q3 - Q1
limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR
print(f"\nLímite inferior: {limite_inf:.2f} kg | Límite superior: {limite_sup:.2f} kg")

# Identificar outliers
es_outlier = (df['peso_paciente_kg'] < limite_inf) | (df['peso_paciente_kg'] > limite_sup)
print(f"Outliers detectados: {es_outlier.sum()}")

# Imputar outliers y NaN con la mediana
mediana_peso = df['peso_paciente_kg'].median()
df.loc[es_outlier, 'peso_paciente_kg'] = mediana_peso
df['peso_paciente_kg'] = df['peso_paciente_kg'].fillna(mediana_peso)

# Distribución después
print("\nDistribución DESPUÉS del tratamiento de outliers:")
print(df['peso_paciente_kg'].describe())

Distribución ANTES del tratamiento de outliers:
count    500.000000
mean      70.173836
std       40.429581
min        2.000000
25%       60.299600
50%       67.482517
75%       76.040456
max      600.000000
Name: peso_paciente_kg, dtype: float64

Límite inferior: 36.69 kg | Límite superior: 99.65 kg
Outliers detectados: 14

Distribución DESPUÉS del tratamiento de outliers:
count    520.000000
mean      68.023037
std       11.736393
min       37.008510
25%       60.950299
50%       67.482517
75%       75.280365
max       98.654367
Name: peso_paciente_kg, dtype: float64


### 8. Manejo de duplicados
- **Duplicados exactos:** mismas filas en todas las columnas.
- **Duplicados parciales:** mismo `id_paciente`, `fecha_prescripcion` y `medicamento` (representa la misma prescripción registrada dos veces).

In [212]:
# Duplicados exactos
n_exactos = df.duplicated().sum()
df = df.drop_duplicates()
print(f"Duplicados exactos eliminados: {n_exactos}")

# Duplicados parciales (misma prescripción)
n_parciales = df.duplicated(
    subset=['id_paciente', 'fecha_prescripcion', 'medicamento']
).sum()
df = df.drop_duplicates(subset=['id_paciente', 'fecha_prescripcion', 'medicamento'])
print(f"Duplicados parciales eliminados: {n_parciales}")
print(f"Filas restantes: {len(df)}")

Duplicados exactos eliminados: 20
Duplicados parciales eliminados: 0
Filas restantes: 500


### 9. Unificación de variantes en `eps`
Se reduce la cardinalidad mediante un diccionario de mapeo manual

In [213]:
print("=== DIAGNÓSTICO INICIAL DE eps ===")
print(f"\nTotal filas: {len(df)}")
print(f"Cardinalidad: {df['eps'].nunique(dropna=False)}")
print(f"Nulos: {df['eps'].isna().sum()}")

print(f"\n--- Valores únicos con frecuencia ---")
print(df['eps'].value_counts(dropna=False))

=== DIAGNÓSTICO INICIAL DE eps ===

Total filas: 500
Cardinalidad: 11
Nulos: 0

--- Valores únicos con frecuencia ---
eps
NuevaEPS       54
SURA           53
COMPENSAR      51
EPS Sura       49
Compensar      49
Nueva EPS      48
Sura           45
Sanitas        39
SANITAS        38
EPS Sanitas    37
nueva eps      37
Name: count, dtype: int64


In [214]:
# Guardar copia del original para auditoría
df['eps_original'] = df['eps'].copy()

# Normalización básica
df['eps'] = df['eps'].astype(str).str.strip().str.lower()

# Mapeo solo de variantes que necesitan unificación
mapeo_eps = {
    'eps sura': 'sura',
    'eps sanitas': 'sanitas',
    'nuevaeps': 'nueva eps',
}
df['eps'] = df['eps'].replace(mapeo_eps)

# Capitalizar a forma canónica
df['eps'] = df['eps'].str.title()

# Verificación final
print("=== RESULTADO FINAL ===")
print(f"\nEPS — {df['eps'].nunique()} valores únicos:")
print(df['eps'].value_counts(dropna=False))

print(f"\n--- Reducción de cardinalidad ---")
print(f"EPS: {df['eps_original'].nunique()} → {df['eps'].nunique()}")

# Validación de cobertura
eps_esperadas = {'Sura', 'Sanitas', 'Compensar', 'Nueva Eps'}
inesperadas = set(df['eps'].dropna().unique()) - eps_esperadas
print(f"\nEPS sin mapear: {inesperadas if inesperadas else 'ninguna ✓'}")

=== RESULTADO FINAL ===

EPS — 4 valores únicos:
eps
Sura         147
Nueva Eps    139
Sanitas      114
Compensar    100
Name: count, dtype: int64

--- Reducción de cardinalidad ---
EPS: 11 → 4

EPS sin mapear: ninguna ✓


## Nivel 3 — Transformación y validación

### 10. Creación de `rango_etario`
Se categoriza la edad en `pediátrico` (<18), `adulto` (18–64) y `adulto mayor` (≥65).  
Las edades inválidas (negativas o > 120) y los `NaN` se etiquetan como `desconocido` para preservar el registro sin sesgar el análisis.

In [215]:
def clasificar_edad(edad):
    """Clasifica la edad en rangos etarios; valores inválidos -> 'desconocido'."""
    if pd.isna(edad) or edad < 0 or edad > 120:
        return 'desconocido'
    if edad < 18:
        return 'pediátrico'
    elif edad < 65:
        return 'adulto'
    else:
        return 'adulto mayor'

df['rango_etario'] = df['edad'].apply(clasificar_edad)
print(df['rango_etario'].value_counts())

rango_etario
adulto          330
adulto mayor    147
desconocido      22
pediátrico        1
Name: count, dtype: int64


### 11. Tabla resumen agrupada por `eps` y `via_administracion`
Incluye el conteo de prescripciones y estadística descriptiva de la `dosis_mg` ya limpia.

In [216]:
# GroupBy con conteo y estadísticas descriptivas
tabla_resumen = df.groupby(['eps', 'via_administracion']).agg(
    n_prescripciones=('dosis_mg', 'count'),
    dosis_media=('dosis_mg', 'mean'),
    dosis_mediana=('dosis_mg', 'median'),
    dosis_std=('dosis_mg', 'std'),
    dosis_min=('dosis_mg', 'min'),
    dosis_max=('dosis_mg', 'max')
).round(2)

tabla_resumen

n_prescripciones  dosis_media  dosis_mediana  \
eps       via_administracion                                                 
Compensar intravenosa                       36       462.50          400.0   
          oral                              40       393.75          400.0   
          subcutanea                        24       533.33          450.0   
Nueva Eps intravenosa                       42       519.05          400.0   
          oral                              48       466.67          400.0   
          subcutanea                        49       482.65          400.0   
Sanitas   intravenosa                       45       392.22          400.0   
          oral                              37       485.14          400.0   
          subcutanea                        32       509.38          500.0   
Sura      intravenosa                       58       507.76          400.0   
          oral                              55       460.91          400.0   
          subcutanea                        34       345.59          400.0   

                              dosis_std  dosis_min  dosis_max  
eps       via_administracion                                   
Compensar intravenosa            267.63      100.0     1000.0  
          oral                   249.41      100.0     1000.0  
          subcutanea             335.79      100.0     1000.0  
Nueva Eps intravenosa            313.51      100.0     1000.0  
          oral                   291.43      100.0     1000.0  
          subcutanea             335.50      100.0     1000.0  
Sanitas   intravenosa            273.85      100.0     1000.0  
          oral                   289.12      100.0     1000.0  
          subcutanea             302.26      100.0     1000.0  
Sura      intravenosa            310.46      100.0     1000.0  
          oral                   298.25      100.0     1000.0  
          subcutanea             224.07      100.0     1000.0

### Normalización de `diagnostico_cie10`
La columna mezcla códigos CIE-10 válidos (`J18.9`, `I10`, `E11`, etc.) con texto libre (`"diabetes tipo 2"`, `"hipertension"`) y `NaN`.  
**Estrategia:** mapear el texto libre a su código CIE-10 equivalente y conservar los `NaN` como `'sin_diagnostico'` para preservar el registro (eliminar filas por falta de diagnóstico descartaría información de prescripción válida).

| Texto libre | Código CIE-10 |
|---|---|
| `diabetes tipo 2` | `E11` |
| `hipertension` | `I10` |

In [217]:
print("Valores únicos ANTES:", df['diagnostico_cie10'].nunique(dropna=False))
print(df['diagnostico_cie10'].unique())

# Normalizar a minúsculas y quitar espacios para comparar texto libre
diag_norm = df['diagnostico_cie10'].astype(str).str.strip().str.lower()

# Mapeo de texto libre a código CIE-10
mapeo_diag = {
    'diabetes tipo 2': 'E11',
    'hipertension': 'I10'
}
diag_norm = diag_norm.replace(mapeo_diag)

# Restaurar mayúsculas en los códigos CIE-10 (que llevan letra inicial mayúscula)
df['diagnostico_cie10'] = diag_norm.str.upper()

# Reemplazar NaN (que quedaron como 'NAN' tras astype(str)) por una etiqueta clara
df['diagnostico_cie10'] = df['diagnostico_cie10'].replace('NAN', 'sin_diagnostico')

print("\nValores únicos DESPUÉS:", df['diagnostico_cie10'].nunique())
print(df['diagnostico_cie10'].unique())

Valores únicos ANTES: 9
[nan 'diabetes tipo 2' 'hipertension' 'J18.9' 'I10' 'J06.9' 'Z00.0'
 'K29.5' 'E11']

Valores únicos DESPUÉS: 7
['sin_diagnostico' 'E11' 'I10' 'J18.9' 'J06.9' 'Z00.0' 'K29.5']


In [218]:
print("=== DIAGNÓSTICO DE prescriptor_id ===")
print(f"Cardinalidad: {df['prescriptor_id'].nunique()}")
print(f"Nulos: {df['prescriptor_id'].isna().sum()}")
print(f"\nMuestra de valores:")
print(df['prescriptor_id'].value_counts().head(10))

=== DIAGNÓSTICO DE prescriptor_id ===
Cardinalidad: 437
Nulos: 53

Muestra de valores:
prescriptor_id
MED-9700    2
MED-2825    2
MED-8994    2
MED-2680    2
MED-6733    2
MED-0358    2
MED-1740    2
MED-2038    2
MED-6359    2
MED-7523    2
Name: count, dtype: int64


In [219]:
#Etiquetar nulos
df['prescriptor_id'] = df['prescriptor_id'].fillna('SIN_REGISTRO')

print(f"Nulos restantes: {df['prescriptor_id'].isna().sum()}")
print(f"Frecuencia de SIN_REGISTRO: {(df['prescriptor_id'] == 'SIN_REGISTRO').sum()}")

Nulos restantes: 0
Frecuencia de SIN_REGISTRO: 53


In [220]:
df.columns

Index(['id_paciente', 'fecha_prescripcion', 'medicamento', 'dosis_mg',
       'via_administracion', 'diagnostico_cie10', 'peso_paciente_kg', 'edad',
       'eps', 'prescriptor_id', 'fecha_original', 'medicamento_original',
       'via_original', 'dosis_original', 'eps_original', 'rango_etario'],
      dtype='object')

In [221]:
# Lista de columnas auxiliares que ya no se necesitan
columnas_auxiliares = [
    'fecha_original',
    'medicamento_original',
    'via_original',
    'dosis_original',
    'eps_original'
]

# Verificar que existen antes de borrar (defensivo)
columnas_a_borrar = [col for col in columnas_auxiliares if col in df.columns]

# Borrar
df = df.drop(columns=columnas_a_borrar)

print(f"Columnas eliminadas: {columnas_a_borrar}")
print(f"Forma del dataset: {df.shape}")
print(f"\nColumnas restantes:")
print(df.columns.tolist())

Columnas eliminadas: ['fecha_original', 'medicamento_original', 'via_original', 'dosis_original', 'eps_original']
Forma del dataset: (500, 11)

Columnas restantes:
['id_paciente', 'fecha_prescripcion', 'medicamento', 'dosis_mg', 'via_administracion', 'diagnostico_cie10', 'peso_paciente_kg', 'edad', 'eps', 'prescriptor_id', 'rango_etario']


In [222]:
# Forma del dataset (filas, columnas)
print("Forma del dataset:", df.shape)
print("\nNombres de columnas:")
print(df.columns.tolist())


# Tipos de dato por columna
print("\nTipos de dato:")
print(df.dtypes)


Forma del dataset: (500, 11)

Nombres de columnas:
['id_paciente', 'fecha_prescripcion', 'medicamento', 'dosis_mg', 'via_administracion', 'diagnostico_cie10', 'peso_paciente_kg', 'edad', 'eps', 'prescriptor_id', 'rango_etario']

Tipos de dato:
id_paciente                    int64
fecha_prescripcion    datetime64[ns]
medicamento                   object
dosis_mg                     float64
via_administracion            object
diagnostico_cie10             object
peso_paciente_kg             float64
edad                         float64
eps                           object
prescriptor_id                object
rango_etario                  object
dtype: object


In [223]:
# Conteo y porcentaje de NaN por columna
faltantes = df.isnull().sum()
porcentaje = (df.isnull().sum() / len(df) * 100).round(2)

# Tabla resumen de completitud
inventario_na = pd.DataFrame({
    'n_faltantes': faltantes,
    'porcentaje_%': porcentaje,
    'completitud_%': (100 - porcentaje).round(2)
}).sort_values('n_faltantes', ascending=False)

inventario_na

,n_faltantes,porcentaje_%,completitud_%
edad,15,3.0,97.0
fecha_prescripcion,0,0.0,100.0
id_paciente,0,0.0,100.0
medicamento,0,0.0,100.0
dosis_mg,0,0.0,100.0
diagnostico_cie10,0,0.0,100.0
via_administracion,0,0.0,100.0
peso_paciente_kg,0,0.0,100.0
eps,0,0.0,100.0
prescriptor_id,0,0.0,100.0


### 12. Exportación del CSV limpio
Se exporta el DataFrame final sin la columna de índice.

In [224]:

# Validación final del DataFrame limpio
print("Forma final:", df.shape)
print("Tipos finales:")
print(df.dtypes)

# Exportar a CSV sin índice
nombre_archivo = 'prescripciones_limpias_ojeda_pardo.csv'
df.to_csv(nombre_archivo, index=False)
print(f"\nArchivo exportado: {nombre_archivo}")

# Descargar el archivo desde Colab al equipo
files.download(nombre_archivo)

Forma final: (500, 11)
Tipos finales:
id_paciente                    int64
fecha_prescripcion    datetime64[ns]
medicamento                   object
dosis_mg                     float64
via_administracion            object
diagnostico_cie10             object
peso_paciente_kg             float64
edad                         float64
eps                           object
prescriptor_id                object
rango_etario                  object
dtype: object

Archivo exportado: prescripciones_limpias_ojeda_pardo.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>